# Copyright and AI: Legal Risk for AI Engineers

AI raises three distinct copyright questions that every engineer must understand before shipping a product:

1. **Can you use copyrighted data to train an AI?** -- Actively litigated; legal risk is real.
2. **Can AI-generated content be copyrighted?** -- No, without sufficient human authorship.
3. **Who owns the model weights?** -- The license chosen by the creator governs.

This notebook covers the legal landscape, model licensing, and risk mitigation in practice.

In [1]:
import json
import hashlib
from datetime import datetime
from typing import List, Optional, Dict, Any
from enum import Enum
from dataclasses import dataclass, field

## Section 1: Training Data and Copyright

### The Legal Landscape

Several major lawsuits are actively shaping the law around AI training data:

| Case | Plaintiff | Defendant | Core Claim |
|------|-----------|-----------|------------|
| NYT vs OpenAI | New York Times | OpenAI / Microsoft | Training on articles without license |
| Getty vs Stability AI | Getty Images | Stability AI | Training on 12M+ images without license |
| Authors Guild vs OpenAI | Authors Guild | OpenAI | Training on copyrighted books |
| Andersen vs Stability AI | Artists | Stability AI, Midjourney | Training on artist styles |

None of these cases has reached final judgment. The legal outcome will determine the scope of fair use for AI training.

### Fair Use Analysis (US Law)

US copyright law allows unauthorized use of copyrighted works for "fair use," evaluated on four factors:

1. **Purpose and character** -- Is the use transformative? Commercial use weighs against fair use.
2. **Nature of the work** -- Factual works get less protection than creative works.
3. **Amount used** -- The more of the work copied, the less likely fair use.
4. **Market effect** -- Does the use harm the market for the original?

AI training may fare well on factor 1 (transformative) but poorly on factor 4 (market substitution) when the output can replace the original.

In [2]:
class LicenseType(str, Enum):
    PUBLIC_DOMAIN = "public_domain"
    CREATIVE_COMMONS_0 = "cc0"  # Waived copyright
    APACHE_2 = "apache_2"       # Very permissive, commercial OK
    CC_BY = "cc_by"             # Attribution required
    CC_BY_SA = "cc_by_sa"       # Attribution + ShareAlike
    CC_BY_NC = "cc_by_nc"       # Non-commercial only
    CUSTOM_RESTRICTED = "custom_restricted"  # Model-specific restrictions
    ALL_RIGHTS_RESERVED = "all_rights_reserved"  # No reuse without permission
    UNKNOWN = "unknown"


@dataclass
class DataSource:
    name: str
    url: str
    license_type: LicenseType
    requires_attribution: bool
    commercial_use_allowed: bool
    training_use_allowed: bool
    notes: str = ""
    added_date: str = field(default_factory=lambda: datetime.now().isoformat()[:10])


# Sample training data sources with their license status
training_sources = [
    DataSource(
        name="Common Crawl",
        url="https://commoncrawl.org",
        license_type=LicenseType.UNKNOWN,
        requires_attribution=False,
        commercial_use_allowed=True,
        training_use_allowed=True,
        notes="Web crawl -- underlying content is subject to original site licenses. High legal risk for creative content."
    ),
    DataSource(
        name="Wikipedia",
        url="https://dumps.wikimedia.org",
        license_type=LicenseType.CC_BY_SA,
        requires_attribution=True,
        commercial_use_allowed=True,
        training_use_allowed=True,
        notes="CC BY-SA 4.0. Attribution required. ShareAlike may propagate to derivative works."
    ),
    DataSource(
        name="The Pile (Books3)",
        url="https://pile.eleuther.ai",
        license_type=LicenseType.ALL_RIGHTS_RESERVED,
        requires_attribution=True,
        commercial_use_allowed=False,
        training_use_allowed=False,
        notes="Books3 subset contains copyrighted books scraped without permission. Removed from public distribution."
    ),
    DataSource(
        name="Gutenberg Project",
        url="https://gutenberg.org",
        license_type=LicenseType.PUBLIC_DOMAIN,
        requires_attribution=False,
        commercial_use_allowed=True,
        training_use_allowed=True,
        notes="Pre-1927 works in public domain in the US. Safe to use."
    ),
    DataSource(
        name="Dolma (AI2)",
        url="https://huggingface.co/datasets/allenai/dolma",
        license_type=LicenseType.APACHE_2,
        requires_attribution=False,
        commercial_use_allowed=True,
        training_use_allowed=True,
        notes="Curated pretraining dataset designed for legal clarity. Preferred for commercial training."
    )
]

print("Training Data Sources: License Status")
print("=" * 70)
for source in training_sources:
    commercial = "YES" if source.commercial_use_allowed else "NO"
    training = "YES" if source.training_use_allowed else "NO"
    print(f"\n{source.name}")
    print(f"  License: {source.license_type.value}")
    print(f"  Commercial: {commercial} | Training: {training} | Attribution: {source.requires_attribution}")
    print(f"  Notes: {source.notes}")

Training Data Sources: License Status

Common Crawl
  License: unknown
  Commercial: YES | Training: YES | Attribution: False
  Notes: Web crawl -- underlying content is subject to original site licenses. High legal risk for creative content.

Wikipedia
  License: cc_by_sa
  Commercial: YES | Training: YES | Attribution: True
  Notes: CC BY-SA 4.0. Attribution required. ShareAlike may propagate to derivative works.

The Pile (Books3)
  License: all_rights_reserved
  Commercial: NO | Training: NO | Attribution: True
  Notes: Books3 subset contains copyrighted books scraped without permission. Removed from public distribution.

Gutenberg Project
  License: public_domain
  Commercial: YES | Training: YES | Attribution: False
  Notes: Pre-1927 works in public domain in the US. Safe to use.

Dolma (AI2)
  License: apache_2
  Commercial: YES | Training: YES | Attribution: False
  Notes: Curated pretraining dataset designed for legal clarity. Preferred for commercial training.


## Section 2: AI-Generated Output and Copyright

### US Copyright Office Position

The US Copyright Office has issued clear guidance (2023): **AI-generated content without human authorship is NOT copyrightable**.

Key cases:
- *Thaler v. Perlmutter* (2023): An AI-generated image ("A Recent Entrance to Paradise") was denied copyright registration because it lacked human authorship.
- *Zarya of the Dawn* (2023): A comic book received partial copyright. Text: copyrightable. Midjourney-generated images: not copyrightable.

### Human Authorship Threshold

The Office will register AI-assisted works where there is **sufficient human creative expression**, including:
- Selection and arrangement of AI outputs
- Significant modification of AI outputs
- Adding human-authored text, images, or other elements

Simply writing a prompt is NOT sufficient. The human must make expressive creative choices about the final output.

### EU Approach

The EU similarly requires human creative contribution. The European Court of Justice has interpreted copyright to require the author's "personal intellectual creation," which purely AI-generated content cannot satisfy.

### Practical Impact

If your product generates content purely with AI, you cannot prevent competitors from copying that content verbatim. The content is in the public domain.

In [3]:
def assess_copyright_eligibility(content_description: str, human_contributions: List[str]) -> Dict[str, Any]:
    """Assess whether an AI-assisted work qualifies for copyright protection."""
    strong_contributions = [
        "selected specific outputs from many options",
        "edited or modified the AI output significantly",
        "added original text or imagery",
        "arranged outputs in a creative structure",
        "made iterative revisions with artistic judgment",
        "combined AI output with original human work"
    ]
    weak_contributions = [
        "wrote the prompt",
        "entered text into the AI tool",
        "pressed the generate button",
        "chose from two options"
    ]

    strong_count = sum(
        1 for contribution in human_contributions
        if any(strong.lower() in contribution.lower() for strong in strong_contributions)
    )
    weak_count = sum(
        1 for contribution in human_contributions
        if any(weak.lower() in contribution.lower() for weak in weak_contributions)
    )

    if strong_count >= 2:
        status = "LIKELY COPYRIGHTABLE"
        explanation = "Significant human creative contribution detected. Register the work and document human contributions."
    elif strong_count == 1:
        status = "POSSIBLY COPYRIGHTABLE"
        explanation = "Borderline case. Consult IP counsel. Document all human creative choices."
    else:
        status = "NOT COPYRIGHTABLE"
        explanation = "Insufficient human authorship. Purely AI-generated content is in the public domain."

    return {
        "content": content_description,
        "human_contributions": human_contributions,
        "strong_contribution_count": strong_count,
        "weak_contribution_count": weak_count,
        "copyright_status": status,
        "explanation": explanation
    }


# Test cases
test_cases = [
    {
        "content": "Marketing copy generated by GPT-4",
        "contributions": ["Wrote the prompt", "Pressed the generate button"]
    },
    {
        "content": "Illustrated children's book with AI images",
        "contributions": [
            "Wrote original story text",
            "Selected specific outputs from many options for each page",
            "Arranged outputs in creative structure across 24 pages"
        ]
    },
    {
        "content": "Product description variations",
        "contributions": ["Entered product name into AI tool", "Chose from two options presented"]
    }
]

for tc in test_cases:
    result = assess_copyright_eligibility(tc["content"], tc["contributions"])
    print(f"Content: {result['content']}")
    print(f"  Status: {result['copyright_status']}")
    print(f"  {result['explanation']}")
    print()

Content: Marketing copy generated by GPT-4
  Status: NOT COPYRIGHTABLE
  Insufficient human authorship. Purely AI-generated content is in the public domain.

Content: Illustrated children's book with AI images
  Status: POSSIBLY COPYRIGHTABLE
  Borderline case. Consult IP counsel. Document all human creative choices.

Content: Product description variations
  Status: NOT COPYRIGHTABLE
  Insufficient human authorship. Purely AI-generated content is in the public domain.



## Section 3: Model Weight Licensing

"Open source" in AI often means "open weights" -- the model weights are public but usage may be restricted by a custom license. **Always check the license before commercial deployment.**

### Common License Types

| License | Commercial Use | Attribution | Distribution | Notes |
|---------|---------------|-------------|--------------|-------|
| Apache 2.0 | YES | Required | Allowed | Most permissive; used by BERT, T5, many HF models |
| MIT | YES | Required | Allowed | Very permissive |
| CC BY 4.0 | YES | Required | Allowed | Common for data/research models |
| CC BY-NC 4.0 | NO | Required | Allowed | Research only; commercial use prohibited |
| LLaMA 3 | Conditional | Required | Conditional | Commercial use OK under 700M MAU threshold |
| Mistral | Apache 2.0 | Required | Allowed | Permissive |
| Stable Diffusion | OpenRAIL-M | Required | Allowed | Prohibits specific harmful use cases |

In [4]:
class LicenseStatus(str, Enum):
    ALLOWED = "allowed"
    RESTRICTED = "restricted"
    PROHIBITED = "prohibited"
    UNKNOWN = "unknown"
    REQUIRES_REVIEW = "requires_review"


@dataclass
class ModelLicense:
    license_name: str
    commercial_use: bool
    derivatives_allowed: bool
    distribution_allowed: bool
    attribution_required: bool
    special_restrictions: List[str] = field(default_factory=list)
    notes: str = ""


MODEL_LICENSES = {
    "meta-llama/Meta-Llama-3-8B": ModelLicense(
        license_name="Meta Llama 3 Community License",
        commercial_use=True,
        derivatives_allowed=True,
        distribution_allowed=True,
        attribution_required=True,
        special_restrictions=[
            "Prohibited if monthly active users > 700 million (requires separate agreement with Meta)",
            "Must include Meta's required attribution notice in product",
            "Cannot be used to train competing foundation models"
        ],
        notes="Commercial use allowed for most businesses. Review MAU restriction."
    ),
    "mistralai/Mistral-7B-v0.1": ModelLicense(
        license_name="Apache 2.0",
        commercial_use=True,
        derivatives_allowed=True,
        distribution_allowed=True,
        attribution_required=True,
        notes="Highly permissive. Include Apache 2.0 license notice."
    ),
    "bert-base-uncased": ModelLicense(
        license_name="Apache 2.0",
        commercial_use=True,
        derivatives_allowed=True,
        distribution_allowed=True,
        attribution_required=True,
        notes="Google BERT. Apache 2.0. Fully permissive for commercial use."
    ),
    "stabilityai/stable-diffusion-xl-base-1.0": ModelLicense(
        license_name="OpenRAIL++-M",
        commercial_use=True,
        derivatives_allowed=True,
        distribution_allowed=True,
        attribution_required=False,
        special_restrictions=[
            "Cannot generate CSAM",
            "Cannot generate content designed to harm individuals",
            "Must propagate license restrictions to downstream users"
        ],
        notes="OpenRAIL license: commercial use OK but with behavioral restrictions that propagate."
    ),
    "some-research/model-nc": ModelLicense(
        license_name="CC BY-NC 4.0",
        commercial_use=False,
        derivatives_allowed=True,
        distribution_allowed=True,
        attribution_required=True,
        notes="Research use only. DO NOT use in commercial products."
    )
}


class LicenseChecker:
    """Check whether a model license permits a specific use case."""

    def check_model_license(
        self,
        model_name: str,
        use_case: str,
        is_commercial: bool = True,
        will_distribute: bool = False
    ) -> Dict[str, Any]:
        license_info = MODEL_LICENSES.get(model_name)

        if license_info is None:
            return {
                "model": model_name,
                "use_case": use_case,
                "status": LicenseStatus.UNKNOWN,
                "message": "License not found in registry. Research the model's license on Hugging Face or the model's official page before using.",
                "restrictions": []
            }

        issues = []
        if is_commercial and not license_info.commercial_use:
            issues.append(f"License '{license_info.license_name}' PROHIBITS commercial use")
        if will_distribute and not license_info.distribution_allowed:
            issues.append(f"License '{license_info.license_name}' PROHIBITS distribution")

        if issues:
            status = LicenseStatus.PROHIBITED
            message = f"USE PROHIBITED: {'; '.join(issues)}"
        elif license_info.special_restrictions:
            status = LicenseStatus.REQUIRES_REVIEW
            message = f"Conditional use: review special restrictions before deployment."
        else:
            status = LicenseStatus.ALLOWED
            message = f"Use appears permitted under {license_info.license_name}."
            if license_info.attribution_required:
                message += " Attribution required."

        return {
            "model": model_name,
            "license": license_info.license_name,
            "use_case": use_case,
            "is_commercial": is_commercial,
            "status": status,
            "message": message,
            "special_restrictions": license_info.special_restrictions,
            "notes": license_info.notes
        }


checker = LicenseChecker()
checks = [
    ("meta-llama/Meta-Llama-3-8B", "customer service chatbot", True, False),
    ("some-research/model-nc", "product recommendation engine", True, False),
    ("mistralai/Mistral-7B-v0.1", "internal code assistant", True, True),
    ("unknown/mystery-model", "document classification", True, False)
]

for model, use_case, commercial, distribute in checks:
    result = checker.check_model_license(model, use_case, commercial, distribute)
    print(f"Model: {model}")
    print(f"  Use Case: {use_case} | Commercial: {commercial}")
    print(f"  Status: {result['status']}")
    print(f"  {result['message']}")
    if result.get('special_restrictions'):
        for restriction in result['special_restrictions']:
            print(f"  Restriction: {restriction}")
    print()

Model: meta-llama/Meta-Llama-3-8B
  Use Case: customer service chatbot | Commercial: True
  Status: LicenseStatus.REQUIRES_REVIEW
  Conditional use: review special restrictions before deployment.
  Restriction: Prohibited if monthly active users > 700 million (requires separate agreement with Meta)
  Restriction: Must include Meta's required attribution notice in product
  Restriction: Cannot be used to train competing foundation models

Model: some-research/model-nc
  Use Case: product recommendation engine | Commercial: True
  Status: LicenseStatus.PROHIBITED
  USE PROHIBITED: License 'CC BY-NC 4.0' PROHIBITS commercial use

Model: mistralai/Mistral-7B-v0.1
  Use Case: internal code assistant | Commercial: True
  Status: LicenseStatus.ALLOWED
  Use appears permitted under Apache 2.0. Attribution required.

Model: unknown/mystery-model
  Use Case: document classification | Commercial: True
  Status: LicenseStatus.UNKNOWN
  License not found in registry. Research the model's license on

## Section 4: Risk Mitigation in Practice

### Data Provenance Tracking

Every training example should be traceable to its source, including the license under which it was used. This creates an audit trail if you face a copyright claim.

In [5]:
@dataclass
class DataRecord:
    record_id: str
    source_name: str
    source_url: str
    license_type: str
    commercial_use_allowed: bool
    requires_attribution: bool
    content_hash: str  # SHA256 of content -- not the content itself
    added_date: str = field(default_factory=lambda: datetime.now().isoformat()[:10])
    attribution_text: str = ""


class DataProvenanceTracker:
    """Track the provenance and license status of every training example."""

    def __init__(self):
        self._records: List[DataRecord] = []

    def _hash_content(self, content: str) -> str:
        return hashlib.sha256(content.encode()).hexdigest()[:12]

    def add_record(
        self,
        content: str,
        source: DataSource
    ) -> DataRecord:
        record = DataRecord(
            record_id=self._hash_content(content + source.url),
            source_name=source.name,
            source_url=source.url,
            license_type=source.license_type.value,
            commercial_use_allowed=source.commercial_use_allowed,
            requires_attribution=source.requires_attribution,
            content_hash=self._hash_content(content),
            attribution_text=f"Source: {source.name} ({source.url})" if source.requires_attribution else ""
        )
        self._records.append(record)
        return record

    def audit_for_commercial_use(self) -> Dict[str, Any]:
        total = len(self._records)
        risky = [r for r in self._records if not r.commercial_use_allowed]
        unknown_license = [r for r in self._records if r.license_type == "unknown"]

        return {
            "total_records": total,
            "commercial_safe": total - len(risky) - len(unknown_license),
            "prohibited_for_commercial": len(risky),
            "unknown_license": len(unknown_license),
            "prohibited_sources": list({r.source_name for r in risky}),
            "risk_assessment": "HIGH" if risky else ("MEDIUM" if unknown_license else "LOW")
        }

    def generate_attribution_file(self) -> str:
        """Generate ATTRIBUTIONS.txt content for required attributions."""
        lines = ["TRAINING DATA ATTRIBUTIONS", "=" * 40, ""]
        sources_needing_attribution = {
            r.source_name: r.attribution_text
            for r in self._records
            if r.requires_attribution and r.attribution_text
        }
        for name, text in sources_needing_attribution.items():
            lines.append(text)
        return "\n".join(lines)


# Demo
tracker = DataProvenanceTracker()

sample_texts = [
    ("The Battle of Hastings was fought in 1066.", training_sources[3]),   # Gutenberg - Public Domain
    ("Climate change refers to long-term shifts in temperatures.", training_sources[1]),  # Wikipedia - CC BY-SA
    ("Our quarterly earnings exceeded expectations.", training_sources[0])  # Common Crawl - Unknown
]

for content, source in sample_texts:
    record = tracker.add_record(content, source)
    print(f"Added: {content[:50]}... | Source: {source.name} | Commercial OK: {record.commercial_use_allowed}")

print("\nCommercial Use Audit:")
audit = tracker.audit_for_commercial_use()
for k, v in audit.items():
    print(f"  {k}: {v}")

Added: The Battle of Hastings was fought in 1066.... | Source: Gutenberg Project | Commercial OK: True
Added: Climate change refers to long-term shifts in tempe... | Source: Wikipedia | Commercial OK: True
Added: Our quarterly earnings exceeded expectations.... | Source: Common Crawl | Commercial OK: True

Commercial Use Audit:
  total_records: 3
  commercial_safe: 2
  prohibited_for_commercial: 0
  unknown_license: 1
  prohibited_sources: []
  risk_assessment: MEDIUM


### Output Filtering: Detecting Training Data Reproduction

One of the key risks is an AI model reproducing verbatim sections of its training data in outputs. This is sometimes called "memorization." Detecting it requires comparing model outputs against a fingerprint database of training content.

In [6]:
class OutputFilter:
    """Detect when model outputs closely reproduce training data."""

    def __init__(self, ngram_size: int = 8):
        self.ngram_size = ngram_size
        self._fingerprints: Dict[str, str] = {}  # ngram_hash -> source_name

    def _get_ngrams(self, text: str) -> List[str]:
        words = text.lower().split()
        return [
            " ".join(words[i:i + self.ngram_size])
            for i in range(len(words) - self.ngram_size + 1)
        ]

    def index_training_document(self, content: str, source_name: str) -> int:
        """Add a training document's n-gram fingerprints to the index."""
        ngrams = self._get_ngrams(content)
        for ngram in ngrams:
            ngram_hash = hashlib.md5(ngram.encode()).hexdigest()
            self._fingerprints[ngram_hash] = source_name
        return len(ngrams)

    def check_output(self, model_output: str) -> Dict[str, Any]:
        """Check if model output reproduces training data."""
        ngrams = self._get_ngrams(model_output)
        if not ngrams:
            return {"match_rate": 0.0, "flagged_sources": [], "recommendation": "Output too short to check"}

        matches = []
        for ngram in ngrams:
            ngram_hash = hashlib.md5(ngram.encode()).hexdigest()
            if ngram_hash in self._fingerprints:
                matches.append(self._fingerprints[ngram_hash])

        match_rate = len(matches) / len(ngrams)
        flagged_sources = list(set(matches))

        if match_rate > 0.5:
            recommendation = "BLOCK: Output closely reproduces copyrighted training data. Do not serve to user."
        elif match_rate > 0.2:
            recommendation = "REVIEW: Partial match with training data. Consider paraphrasing before serving."
        else:
            recommendation = "PASS: Output does not closely reproduce training data."

        return {
            "match_rate": round(match_rate, 3),
            "total_ngrams": len(ngrams),
            "matched_ngrams": len(matches),
            "flagged_sources": flagged_sources,
            "recommendation": recommendation
        }


# Demo
training_doc = (
    "The quick brown fox jumps over the lazy dog. "
    "This sentence contains every letter of the English alphabet. "
    "It has been used in typography and typewriter tests for over a century."
)

output_filter = OutputFilter(ngram_size=5)
indexed = output_filter.index_training_document(training_doc, "TestCorpus")
print(f"Indexed {indexed} n-grams from training document.")

test_outputs = [
    "The quick brown fox jumps over the lazy dog. This sentence contains every letter of the English alphabet.",
    "AI can help with many tasks including writing, coding, and data analysis.",
    "The lazy dog was sleeping while the brown fox jumped over it in the English countryside."
]

print()
for output in test_outputs:
    result = output_filter.check_output(output)
    print(f"Output: {output[:60]}...")
    print(f"  Match rate: {result['match_rate']:.1%} | {result['recommendation']}")
    print()

Indexed 27 n-grams from training document.

Output: The quick brown fox jumps over the lazy dog. This sentence c...
  Match rate: 100.0% | BLOCK: Output closely reproduces copyrighted training data. Do not serve to user.

Output: AI can help with many tasks including writing, coding, and d...
  Match rate: 0.0% | PASS: Output does not closely reproduce training data.

Output: The lazy dog was sleeping while the brown fox jumped over it...
  Match rate: 0.0% | PASS: Output does not closely reproduce training data.



## Section 5: Practical Guidelines for AI Engineers

### IP Indemnification

Some major AI providers offer IP indemnification -- they cover your legal costs if a copyright claim arises from using their API outputs:

| Provider | Indemnification | Conditions |
|----------|-----------------|------------|
| OpenAI | Yes (Copyright Shield) | Enterprise/paid tiers; must follow usage policies |
| Google | Yes (Gen AI indemnification) | Vertex AI users; must use safety filters |
| Microsoft Azure | Yes | Azure OpenAI Service customers |
| Anthropic | No (as of mid-2025) | Review current terms before deploying |
| Open-source models | No | License only; no legal coverage |

For high-volume commercial use, using a provider with indemnification shifts legal risk to them.

In [7]:
def copyright_risk_checklist() -> None:
    """Print a practical copyright risk checklist for AI engineers."""
    checklist = {
        "Training Data": [
            "All training data sources inventoried with license information",
            "Only commercially-licensed data used for commercial models",
            "Attribution files generated for CC-BY sources",
            "Data provenance tracker implemented",
            "Legal review completed for any 'unknown license' sources"
        ],
        "Model Weights": [
            "License verified for every third-party model used",
            "Commercial use confirmed for all models in production",
            "MAU and other thresholds checked (especially LLaMA-family)",
            "Attribution notices included in product where required",
            "License propagation requirements reviewed for fine-tuned models"
        ],
        "AI-Generated Output": [
            "No copyright claims made over purely AI-generated content",
            "Human authorship contributions documented for AI-assisted works",
            "Output filter deployed to detect training data memorization",
            "IP indemnification provider chosen for high-volume commercial use",
            "Terms of service disclose AI nature of generated content"
        ]
    }

    print("COPYRIGHT RISK MITIGATION CHECKLIST FOR AI ENGINEERS")
    print("=" * 60)
    for category, items in checklist.items():
        print(f"\n[{category}]")
        for item in items:
            print(f"  [ ] {item}")


copyright_risk_checklist()

COPYRIGHT RISK MITIGATION CHECKLIST FOR AI ENGINEERS

[Training Data]
  [ ] All training data sources inventoried with license information
  [ ] Only commercially-licensed data used for commercial models
  [ ] Attribution files generated for CC-BY sources
  [ ] Data provenance tracker implemented
  [ ] Legal review completed for any 'unknown license' sources

[Model Weights]
  [ ] License verified for every third-party model used
  [ ] Commercial use confirmed for all models in production
  [ ] MAU and other thresholds checked (especially LLaMA-family)
  [ ] Attribution notices included in product where required
  [ ] License propagation requirements reviewed for fine-tuned models

[AI-Generated Output]
  [ ] No copyright claims made over purely AI-generated content
  [ ] Human authorship contributions documented for AI-assisted works
  [ ] Output filter deployed to detect training data memorization
  [ ] IP indemnification provider chosen for high-volume commercial use
  [ ] Terms of 

## Key Takeaways

1. **Training data copyright is actively litigated**: Major lawsuits are ongoing. The safest approach is to use permissively licensed or public domain data for training. If using scraped web data, understand the risk.

2. **AI-generated output is NOT copyrightable** in the US and EU without substantial human authorship. You cannot prevent others from copying purely AI-generated text or images.

3. **'Open source' models may not be truly open**: LLaMA, Mistral, and other 'open weights' models have license restrictions. Always read the license before commercial deployment.

4. **Track data provenance**: Log where every training example came from, including the license. This creates a defensible audit trail.

5. **Output filtering is a safety measure**: Detect when models reproduce training data verbatim and block those outputs.

6. **IP indemnification matters for commercial use**: Choose providers that offer copyright shield / indemnification for high-volume commercial deployments.

7. **Add human authorship to AI-assisted creative work**: Selection, arrangement, and significant modification of AI outputs can qualify for copyright protection. Document those creative choices.